In [1]:
import json
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    make_scorer
)

SEED = 42
TEST_SIZE = 0.20

df = pd.read_csv("breast_cancer.csv")
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y
)

numeric_columns = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = X_train.select_dtypes(exclude=["number"]).columns.tolist()

def build_preprocessor():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns)
    ])

models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", build_preprocessor()),
        ("model", LogisticRegression(max_iter=2000, random_state=SEED))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", build_preprocessor()),
        ("model", RandomForestClassifier(
            n_estimators=100,
            random_state=SEED
        ))
    ])
}

for name, model in models.items():
    model.fit(X_train, y_train)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Models trained:", list(models.keys()))

Training rows: 455
Testing rows: 114
Models trained: ['Logistic Regression', 'Random Forest']


In [2]:
experiment_records = []

for name, model in models.items():
    predictions = model.predict(X_test)

    experiment_records.append({
        "experiment_time_utc": datetime.now(timezone.utc).isoformat(),
        "model": name,
        "seed": SEED,
        "test_size": TEST_SIZE,
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "parameters": model.named_steps["model"].get_params(),
        "python_version": platform.python_version(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "sklearn_version": sklearn.__version__,
        "accuracy": accuracy_score(y_test, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
        "malignant_recall": recall_score(
            y_test, predictions, pos_label=0
        ),
        "malignant_precision": precision_score(
            y_test, predictions, pos_label=0, zero_division=0
        ),
        "malignant_f1": f1_score(
            y_test, predictions, pos_label=0, zero_division=0
        )
    })

experiment_log = pd.DataFrame(experiment_records)

display(
    experiment_log[
        ["model", "seed", "accuracy", "balanced_accuracy",
         "malignant_recall", "malignant_precision", "malignant_f1"]
    ].round(4)
)

,model,seed,accuracy,balanced_accuracy,malignant_recall,malignant_precision,malignant_f1
0,Logistic Regression,42,0.9825,0.9812,0.9762,0.9762,0.9762
1,Random Forest,42,0.9561,0.9504,0.9286,0.9512,0.9398


In [3]:
reports_dir = Path("../../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

with open(reports_dir / "day4_experiment_log.json", "w") as file:
    json.dump(experiment_records, file, indent=2, default=str)

print("Experiment log saved.")

Experiment log saved.


In [4]:
error_rows = []

for name, model in models.items():
    predictions = model.predict(X_test)

    for row_index, actual, predicted in zip(
        X_test.index, y_test, predictions
    ):
        if actual != predicted:
            error_rows.append({
                "model": name,
                "row_index": int(row_index),
                "actual_class": int(actual),
                "predicted_class": int(predicted),
                "error_type": (
                    "Missed malignant case"
                    if actual == 0 and predicted == 1
                    else "Benign case predicted malignant"
                    if actual == 1 and predicted == 0
                    else "Other misclassification"
                )
            })

errors_df = pd.DataFrame(error_rows)

if errors_df.empty:
    print("No incorrect predictions on this test set.")
else:
    display(errors_df)

errors_df.to_csv(
    reports_dir / "day4_misclassified_cases.csv",
    index=False
)

print("Misclassified cases saved.")

,model,row_index,actual_class,predicted_class,error_type
0,Logistic Regression,541,1,0,Benign case predicted malignant
1,Logistic Regression,73,0,1,Missed malignant case
2,Random Forest,363,1,0,Benign case predicted malignant
3,Random Forest,406,1,0,Benign case predicted malignant
4,Random Forest,41,0,1,Missed malignant case
5,Random Forest,73,0,1,Missed malignant case
6,Random Forest,385,0,1,Missed malignant case


Misclassified cases saved.


In [5]:
logistic_model = models["Logistic Regression"]

class_values = list(logistic_model.named_steps["model"].classes_)
malignant_column = class_values.index(0)

malignant_probabilities = logistic_model.predict_proba(X_test)[
    :, malignant_column
]

threshold_results = []

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
    # Predict malignant (0) when its probability meets the threshold
    predictions = np.where(
        malignant_probabilities >= threshold, 0, 1
    )

    threshold_results.append({
        "malignant_threshold": threshold,
        "accuracy": accuracy_score(y_test, predictions),
        "malignant_recall": recall_score(
            y_test, predictions, pos_label=0, zero_division=0
        ),
        "malignant_precision": precision_score(
            y_test, predictions, pos_label=0, zero_division=0
        ),
        "malignant_f1": f1_score(
            y_test, predictions, pos_label=0, zero_division=0
        ),
        "missed_malignant_cases": int(
            ((y_test.to_numpy() == 0) & (predictions == 1)).sum()
        )
    })

threshold_table = pd.DataFrame(threshold_results)
display(threshold_table.round(4))

threshold_table.to_csv(
    reports_dir / "day4_threshold_analysis.csv",
    index=False
)

,malignant_threshold,accuracy,malignant_recall,malignant_precision,malignant_f1,missed_malignant_cases
0,0.2,0.9386,0.9762,0.8723,0.9213,1
1,0.3,0.9474,0.9762,0.8913,0.9318,1
2,0.4,0.9561,0.9762,0.9111,0.9425,1
3,0.5,0.9825,0.9762,0.9762,0.9762,1
4,0.6,0.9825,0.9762,0.9762,0.9762,1
5,0.7,0.9825,0.9524,1.0000,0.9756,2


In [6]:
print("Full dataset class proportions:")
print(y.value_counts(normalize=True).sort_index().round(4))

print("\nTest-set class counts:")
print(y_test.value_counts().sort_index())

Full dataset class proportions:
target
0    0.3726
1    0.6274
Name: proportion, dtype: float64

Test-set class counts:
target
0    42
1    72
Name: count, dtype: int64


In [7]:
y_test_malignant = (y_test == 0).astype(int)

brier = brier_score_loss(
    y_test_malignant,
    malignant_probabilities
)

print("Malignant-probability Brier score:", round(brier, 4))

Malignant-probability Brier score: 0.0215


In [8]:
def predict_malignant_at_threshold(probabilities, threshold):
    if not 0 <= threshold <= 1:
        raise ValueError("Threshold must be between 0 and 1.")

    return np.where(probabilities >= threshold, 0, 1)

# Valid boundary cases
assert np.all(
    predict_malignant_at_threshold(
        np.array([0.0, 1.0]), 0.5
    ) == np.array([1, 0])
)

print("Boundary test passed.")

# Invalid threshold should fail
try:
    predict_malignant_at_threshold(malignant_probabilities, 1.5)
    raise AssertionError("Invalid threshold was not rejected.")
except ValueError as error:
    print("Negative test passed:", error)

Boundary test passed.
Negative test passed: Threshold must be between 0 and 1.
